# Experiment Mobile Controller

Load a mobile camera controller configuration and inspect which observation dome latitude/longitude positions are reachable.

In [ ]:
from pathlib import Path
import sys
sys.path.append("..")

from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"

from interbotix_common_modules.common_robot.robot import robot_shutdown, robot_startup
from interbotix_xs_modules.xs_robot.arm import InterbotixManipulatorXS
from mobile_camera.observation_dome import ObservationDome
from mobile_camera.widowx_observation_dome import WidowX_ObservationDomeDriver


experiment = "controllers"
run = "mobile_camera_controller_00"
exp = Config().get_experiment(experiment, run)

dome_config = exp["observation_dome"]
dome = ObservationDome(
    x=dome_config["x"],
    y=dome_config["y"],
    z=dome_config["z"],
    r=dome_config["r"],
    no_long=dome_config["no_long"],
    no_lat=dome_config["no_lat"],
)

bot = InterbotixManipulatorXS(
    robot_model=exp.get("robot_model", "wx250s"),
    group_name=exp.get("group_name", "arm"),
    gripper_name=exp.get("gripper_name", "gripper"),
)
robot_startup()

dome_driver = WidowX_ObservationDomeDriver(
    bot.arm,
    dome,
    safety_box=exp.get("safety_box"),
)
reachability_kwargs = exp.get("decision", {}).get("reachability_kwargs", {})


def reachable_report(lat_index, long_index):
    return dome_driver.is_lat_long_reachable(
        lat_index,
        long_index,
        return_report=True,
        **reachability_kwargs,
    )


try:
    reachable_positions = []
    print(f"Loaded exp/run {experiment}/{run}: {exp.get('controller_name', '<unnamed controller>')}")
    print("Reachable lat/long positions:")

    for lat_index in range(dome.no_lat + 1):
        long_indices = [0] if lat_index == 0 else range(dome.no_long)
        for long_index in long_indices:
            report = reachable_report(lat_index, long_index)
            if report["safe"] and report["reachable"]:
                reachable_positions.append((lat_index, long_index))
                x, y, z = dome.get_lat_long_point(lat_index, long_index)
                print(f"  lat={lat_index:2d}, long={long_index:2d} -> x={x:.4f}, y={y:.4f}, z={z:.4f}")

    if not reachable_positions:
        print("  <none>")

    default_position = (
        exp.get("default_lat_index", 0),
        exp.get("default_long_index", 0),
    )
    decision_config = exp.get("decision", {})
    middle_position = (
        decision_config.get("middle_lat_index", dome.no_lat // 2),
        decision_config.get("middle_long_index", 0),
    )

    for label, position in (
        ("default", default_position),
        ("middle", middle_position),
    ):
        lat_index, long_index = position
        report = reachable_report(lat_index, long_index)
        reachable = report["safe"] and report["reachable"]
        print(f"{label.title()} position lat={lat_index}, long={long_index} reachable: {reachable} ({report['status']})")
finally:
    robot_shutdown(bot.core.get_node())
